In [1]:
%load_ext autoreload
%autoreload 2

from functools import partial

from absl.testing import parameterized
import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
import numpy as np

import moe
from moe.core import run_moe, add_indices
from moe.ra2a_simulator import ragged_all_to_all as ra2a_via_ag
from moe.pipelined import create_moe
from tests.utils import generate_data

# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

try:
  jax.config.update("jax_num_cpu_devices", 8)
except RuntimeError:
  pass

random_normal = lambda key, shape, dtype: jnp.array(np.random.default_rng(key).normal(size=shape)).astype(dtype)
random_randint = lambda key, shape, minval, maxval: jnp.array(np.random.default_rng(key).integers(
    minval, maxval, size=shape
)).astype(jnp.int32)

In [2]:
experts_per_tok = 4
multiple = 1
devices = jax.devices()
axis_name = "x"
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

jax.sharding.set_mesh(mesh)

In [3]:
#n, k, g = 1024, 128, 32
m, k, g = 4096 * 8, 7168, 32
x = jax.jit(lambda: jax.random.normal(jax.random.key(0), (m, k), dtype="bfloat16"), out_shardings=P("x", None))()
all_idxs = jax.jit(lambda: jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g), out_shardings=P(None))()
x = x.reshape((x.shape[0], 8, -1))

In [4]:
def compute_block(y, group_sizes):
  # construct dummy weight where the weight is just the expert index
  shard_idx = jax.lax.axis_index(axis_name)
  iota = jax.lax.broadcasted_iota("int32", (y.shape[0], group_sizes.shape[-1]), 0)
  starts, ends = jnp.cumsum(group_sizes) - group_sizes, jnp.cumsum(group_sizes)
  assert (g // len(devices)) == group_sizes.size
  group_idxs = group_sizes.size * shard_idx + jnp.arange(group_sizes.size)
  weights = jnp.sum(((iota >= starts[None, :]) & (iota < ends[None, :])) * group_idxs[None, :], -1)
  return y * weights[:, None, None]

In [5]:
opts = dict(ragged_all_to_all=jax.lax.ragged_all_to_all, axis_name=axis_name, experts_num=g, gathers="builtin")
moe_fn = jax.jit(partial(run_moe, compute_block=compute_block, **opts))

In [6]:
y_ref = moe_fn(x, all_idxs)

In [7]:
jnp.sum(y_ref)  # to detect out-of-bounds

Array(-1.79306e+08, dtype=bfloat16)

In [8]:
@jax.jit
@partial(jax.shard_map, in_specs=((P(axis_name, None, None)), P()), out_specs=P(axis_name, None, None), check_vma=False)
def custom_moe(x, all_idxs):
  opts = dict(axis_name=axis_name, experts_per_tok=experts_per_tok, experts_num=g, gathers="custom_sc")
  #moe_methods = create_moe(compute_block, ragged_all_to_all=jax.lax.ragged_all_to_all, **opts)
  moe_methods = create_moe(compute_block, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), **opts)
  meta = moe_methods.compute_meta(all_idxs)
  y = moe_methods.load_fn(x, meta)
  y = moe_methods.compute_fn(y, meta)
  return moe_methods.unload_fn(y, meta)

In [10]:
y = custom_moe(x, all_idxs)

In [11]:
jnp.mean(jnp.abs(y_ref - y) == 0)

Array(1., dtype=float32)

In [12]:
y = jax.block_until_ready(custom_moe(x, all_idxs))
#y_ref = jax.block_until_ready(moe_fn(x, all_idxs))
with moe.utils.profile():
  for _ in range(3):
    jax.block_until_ready(custom_moe(x, all_idxs))
  #for _ in range(4):
  #  jax.block_until_ready(moe_fn(x, all_idxs))

http://localhost:52435/data/plugin/profile/trace_viewer@;run=2025_11_28_03_34_40;tag=trace_viewer@


In [13]:
#opts = dict(axis_name="x", experts_num=g, ragged_all_to_all=partial(moe.sc_kernels.ra2a, multiple=1), multiple=multiple, compute_block=compute_block)
opts = dict(axis_name="x", experts_num=g, ragged_all_to_all=jax.lax.ragged_all_to_all, multiple=multiple, compute_block=compute_block)
moe1_fn = jax.jit(partial(run_moe, **opts, gathers="builtin"))
moe2_fn = jax.jit(partial(run_moe, **opts, gathers="custom_sc"))
#moe2_fn = jax.jit(partial(run_moe, **opts, gathers="custom"))

o1, vjp1_fn = jax.vjp(partial(moe1_fn, all_idxs=all_idxs), x)
o2, vjp2_fn = jax.vjp(partial(moe2_fn, all_idxs=all_idxs), x)
vjp1_fn, vjp2_fn = jax.jit(vjp1_fn), jax.jit(vjp2_fn)

In [14]:
# np.testing.assert_allclose(x, x_new)
x_ref = jnp.repeat(x, experts_per_tok, axis=0, out_sharding=P(axis_name, None))
x_ref = x_ref.reshape((x.shape[0], experts_per_tok, *x.shape[1:]))
x_ref *= all_idxs.reshape((x.shape[0], experts_per_tok, 1, 1))
x_ref = jnp.sum(x_ref, 1)
np.testing.assert_allclose(o1, o2)
np.testing.assert_allclose(x_ref, o1)

In [15]:
r = jax.jit(lambda: jax.random.normal(jax.random.key(1), o1.shape, dtype=x.dtype),
            out_shardings=P(axis_name, None, None))()
(do1,) = vjp1_fn(r)
(do2,) = vjp2_fn(r)

In [18]:
do1_error = jnp.max(jnp.linalg.norm(do1 - do2, axis=-1) / jnp.maximum(jnp.linalg.norm(do1, axis=-1), 1e-7))
assert do1_error < 5e-3

In [ ]:
with moe.utils.profile():
  for _ in range(2):
    jax.block_until_ready(moe1_fn(x, all_idxs))
  for _ in range(2):
    jax.block_until_ready(moe2_fn(x, all_idxs))
  for _ in range(2):
    jax.block_until_ready(vjp1_fn(r))
  for _ in range(2):
    jax.block_until_ready(vjp2_fn(r))

# remaining tests

In [ ]:
@parameterized.product(experts_per_tok=[1, 2, 4, 8], device=["cpu", "tpu"])
def test_simple_moe(self, experts_per_tok, device):
  try:
    devices = jax.devices(device)
  except RuntimeError:
    self.skipTest(f"Device {device} not available")
  axis_name = "x"
  mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

  with jax.sharding.set_mesh(mesh):
    n, k, g = 256, 2048, 32
    # x, ra2a_meta = generate_data(n, k, len(devices), axis_name="x")
    # del ra2a_meta
    x = jax.random.normal(jax.random.key(0), (n, k), dtype="bfloat16")
    all_idxs = jax.random.randint(jax.random.key(0), (experts_per_tok * x.shape[0],), minval=0, maxval=g)
    x, all_idxs = jax.device_put(x, P(axis_name, None)), jax.device_put(all_idxs, P(None))
    out = run_moe(x, all_idxs, axis_name="x", experts_num=g, ragged_all_to_all=ra2a_via_ag)
    self.assertEqual(out.shape, (n, experts_per_tok, x.shape[-1]))

    x_new = np.array(out[:, 0, :])
    np.testing.assert_allclose(x, x_new)

In [ ]:
@parameterized.product(experts=[32, 128], multiple=[2, 4, 8])
def test_add_indices_works_for_moe(self, experts, multiple):
  all_idxs = jax.random.randint(jax.random.key(0), 128, minval=0, maxval=experts)
  idx_count = jnp.bincount(all_idxs, length=experts)
  pad_indices = add_indices(jnp.arange(experts), -idx_count % multiple, max_size=multiple - 1)
  # check if the pad_indices actually added the desired number of pad indices to each group
  np.testing.assert_array_equal(jnp.bincount(pad_indices, length=experts), -idx_count % multiple)

In [ ]:
@parameterized.product(experts_per_tok=[1, 2, 4], device=["cpu", "tpu"], multiple=[1, 2, 8])
def test_identity_moe_block(self, experts_per_tok, device, multiple):
  try:
    devices = jax.devices(device)
  except RuntimeError:
    self.skipTest(f"Device {device} not available")
  axis_name = "x"
  m, k, g = 4096, 128, 32
  mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)

  with jax.sharding.set_mesh(mesh):
    all_idxs = jax.random.randint(jax.random.key(0), experts_per_tok * m, minval=0, maxval=g)
    x, ra2a_meta = generate_data(m, k, device_num=len(devices), axis_name=axis_name)
    del ra2a_meta
    reduce_block = lambda x: x[:, 0, ...]
    moe_fn = jax.jit(partial(run_moe, reduce_block=reduce_block, axis_name=axis_name, experts_num=g,
                              multiple=multiple, ragged_all_to_all=ra2a_via_ag))
    out = moe_fn(x, all_idxs)
    self.assertEqual(out.shape, (m, x.shape[-1]))
    np.testing.assert_array_equal(out, x)